In [115]:
from deck import full_euchre_deck
from dealer import Dealer
from n_game_sim import n_game_sim
from n_play_round import round1, n_play_round, next_round
from n_branches import array_set_difference, n_find_winner, array_set_difference
from tree_search import find_best_opener, trick_played #, r2_best_response
import numpy as np
from numba import njit

In [116]:
game = Dealer(deck=full_euchre_deck ,players=4)
# game.stack_deck(stack_cards=dealer_pick, player=4)
game.deal_cards()
hands5 = np.array([game.hand1, game.hand2, game.hand3, game.hand4])
hands5

array([[[-12,   0],
        [-13,   0],
        [ 12,   0],
        [-10,   0],
        [  0, -12]],

       [[  0, 110],
        [  0,  -9],
        [  0, 120],
        [  0, 140],
        [ 13,   0]],

       [[  0, 130],
        [ 11,   0],
        [ 10,   0],
        [  0, -13],
        [  0, 135]],

       [[-11,   0],
        [ 14,   0],
        [ -9,   0],
        [  0, 100],
        [  0, -14]]])

In [146]:
for i in range(4,6):
    print(i)

4
5


In [165]:
def n_trick_sim(
    game_hand: np.ndarray, 
    r1_chosen_card: np.ndarray, 
    lead: int,
    num_tricks: int,
    previous_winners: np.array
):
    """
    Simulates a game with a specified number of tricks and evaluates the outcome.

    Arguments:
        game_hand (numpy.ndarray): A 3D array representing the cards dealt to each player.
        r1_chosen_card (np.ndarray): The index of the card chosen for the first round.
        lead (int): The player who leads the first trick.
        num_tricks (int): Number of tricks to simulate (3, 4, or 5). Default is 5.
        previous_winners (np.ndarray): Optional array of previous trick winners to include in scoring.

    Returns:
        float: The mean performance evaluation, indicating win probability.
    """
    
    print(num_tricks)

    # First round
    next_leads, next_hands = round1(
        hands_dealt=game_hand, chosen_card=r1_chosen_card, leader=lead
    )
    
    # Determine starting round number based on num_tricks
    # If simulating from round 1, the loop starts at round 2, since we already sim the first round. 

    if num_tricks >= 4:
        start_round = 6 - num_tricks + 1
        end_round = num_tricks + 1

    else:
        start_round = 6 - num_tricks + 1
        end_round = 6


    
    # Simulate subsequent rounds
    current_hands = next_hands
    current_leads = next_leads
    current_score = next_leads.reshape(-1, 1)


    print(start_round, end_round)
    
    for round_num in range(start_round, end_round):
        current_leads, current_hands, current_score = next_round(
            current_hands=current_hands,
            leads=current_leads,
            game_round=round_num,
            game_score=current_score
        )
    
    # Reshape results
    print(current_score)

    # since we add back in the previous winners, we can always treat this as a 5 round trick
    results = current_score.reshape(current_score.shape[0], num_tricks)
    
    # Add previous winners to results if they exist
    # if len(previous_winners) > 0:
    #     for i in range(results.shape[0]):
    #         # The results array already has the current tricks
    #         # We need to account for previous winners in the scoring
    #         pass  # We'll handle this in the scoring logic below
    
    # Calculate meta results
    meta_results = np.zeros(results.shape[0], dtype=np.int64)
    
    for i in range(len(results)):
        if np.sum(results[i] % 2) + np.sum(previous_winners % 2) >= 3:
            score = 1
        else: 
            score = -1

        meta_results[i] = score


    return np.mean(meta_results)

In [166]:
best_opener = find_best_opener(hands=hands5, lead=0, tricks=5, previous_winners=np.array([]), sim_func=n_trick_sim)

5
2 6
[[[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [2]
  [1]]

 ...

 [[1]
  [3]
  [3]
  [1]
  [2]]

 [[1]
  [3]
  [3]
  [2]
  [1]]

 [[1]
  [3]
  [3]
  [2]
  [1]]]
5
2 6
[[[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [2]
  [1]]

 ...

 [[1]
  [3]
  [3]
  [1]
  [2]]

 [[1]
  [3]
  [3]
  [2]
  [1]]

 [[1]
  [3]
  [3]
  [2]
  [1]]]
5
2 6
[[[3]
  [2]
  [2]
  [1]
  [1]]

 [[3]
  [2]
  [2]
  [1]
  [2]]

 [[3]
  [2]
  [2]
  [2]
  [1]]

 ...

 [[3]
  [3]
  [1]
  [2]
  [1]]

 [[3]
  [3]
  [1]
  [2]
  [1]]

 [[3]
  [3]
  [1]
  [2]
  [1]]]
5
2 6
[[[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [1]
  [1]]

 [[2]
  [3]
  [0]
  [2]
  [1]]

 ...

 [[1]
  [3]
  [3]
  [1]
  [2]]

 [[1]
  [3]
  [3]
  [2]
  [1]]

 [[1]
  [3]
  [3]
  [2]
  [1]]]
5
2 6
[[[3]
  [2]
  [3]
  [1]
  [1]]

 [[3]
  [2]
  [3]
  [2]
  [1]]

 [[3]
  [2]
  [3]
  [1]
  [2]]

 ...

 [[3]
  [1]
  [3]
  [1]
  [2]]

 [[3]
  [1]
  [3]
  [2]
  [1]]

 [[3]
  [1]


In [167]:
r2_leads, r2_hands = round1(hands_dealt=hands5, chosen_card=best_opener, leader=0)

In [168]:
# find the best response by iterating through each possible trick from the perspective of the opponent
# need to make this into a function as well

# team 1 (i.e. positions 1 and 3) should go for the best result
responder=1

r1_response_res = np.zeros(r2_leads.shape, dtype=np.float64)

for w in range(r2_leads.shape[0]):

    r3_leads, r3_hands, r2_score = next_round(
        current_hands=np.array([r2_hands[w]]),
        leads=np.array([r2_leads[w]]),
        game_round=2,
        game_score=np.array([r2_leads[w]]).reshape(-1, 1),
    )
    r4_leads, r4_hands, r3_score = next_round(
        current_hands=r3_hands, leads=r3_leads, game_round=3, game_score=r2_score
    )
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    # if eval_position % 2 == 0:
    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2)>=3

    r1_response_res[w] = np.mean(meta_results)

if responder % 2 == 0:
    best_response = np.argmin(r1_response_res)
else:
    best_response = np.argmax(r1_response_res)

print(best_response)

1


In [169]:
trick_played(hands5, r2_hands[best_response])

array([[-12,   0],
       [  0, 110],
       [ 10,   0],
       [ -9,   0]])

In [170]:
r2_leads[best_response]

np.int64(1)

In [171]:
r1_winner = r2_leads[best_response]

In [172]:
r2_optimal =  r2_hands[best_response]

In [173]:
r2_optimal

array([[[-13,   0],
        [ 12,   0],
        [-10,   0],
        [  0, -12]],

       [[  0,  -9],
        [  0, 120],
        [  0, 140],
        [ 13,   0]],

       [[  0, 130],
        [ 11,   0],
        [  0, -13],
        [  0, 135]],

       [[-11,   0],
        [ 14,   0],
        [  0, 100],
        [  0, -14]]])

In [174]:
r2_best_opener = find_best_opener(hands=r2_optimal, lead=r1_winner, tricks=4, previous_winners=np.array([r1_winner]), sim_func=n_trick_sim)

4
3 5
[[[3]
  [0]
  [2]
  [3]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [1]
  [1]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [3]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [3]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [1]
  [3]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [2]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [0]
  [1]]

 [[3]
  [0]
  [2]
  [2]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [2]
  [3]]

 [[3]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [1]
  [1]]

 [[3]


In [175]:
r1_winner

np.int64(1)

In [176]:
r3_leads, r3_hands = round1(hands_dealt=r2_optimal, chosen_card=r2_best_opener, leader=r1_winner)

In [177]:
responder=0

r2_response_res = np.zeros(r3_leads.shape, dtype=np.float64)

for w in range(r3_leads.shape[0]):

    r4_leads, r4_hands, r3_score = next_round(
        current_hands=np.array([r3_hands[w]]),
        leads=np.array([r3_leads[w]]),
        game_round=3,
        game_score=np.array([r3_leads[w]]).reshape(-1, 1),
    )
    
    r5_leads, r5_hands, r4_score = next_round(
        current_hands=r4_hands, leads=r4_leads, game_round=4, game_score=r3_score
    )
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2 + (r1_winner%2))>=3

if responder % 2 == 0:
    best_response = np.argmin(r2_response_res)
else:
    best_response = np.argmax(r2_response_res)
    
print(best_response)

0


In [178]:
trick_played(r2_optimal, r3_hands[best_response])

array([[-10,   0],
       [  0, 140],
       [  0, 130],
       [  0, 100]])

In [179]:
r2_winner = r3_leads[best_response]

In [180]:
r3_optimal =  r3_hands[best_response]

In [181]:
r3_optimal

array([[[-13,   0],
        [ 12,   0],
        [  0, -12]],

       [[  0,  -9],
        [  0, 120],
        [ 13,   0]],

       [[ 11,   0],
        [  0, -13],
        [  0, 135]],

       [[-11,   0],
        [ 14,   0],
        [  0, -14]]])

In [182]:
r3_best_opener = find_best_opener(hands=r3_optimal, lead=r2_winner, tricks=3, previous_winners=np.array([r1_winner, r2_winner]), sim_func=n_trick_sim)

3
4 6
[[[3]
  [0]
  [0]
  [1]
  [2]]

 [[3]
  [0]
  [0]
  [2]
  [3]]

 [[3]
  [0]
  [0]
  [0]
  [2]]

 [[3]
  [0]
  [0]
  [2]
  [1]]

 [[3]
  [0]
  [0]
  [3]
  [2]]]


ValueError: cannot reshape array of size 25 into shape (5,3)

In [ ]:
r2_winner

np.int64(3)

In [ ]:
r4_leads, r4_hands = round1(hands_dealt=r3_optimal, chosen_card=r3_best_opener, leader=r2_winner)

In [ ]:
responder=1

r3_response_res = np.zeros(r4_leads.shape, dtype=np.float64)

for w in range(r4_leads.shape[0]):

    r5_leads, r5_hands, r4_score = next_round(
        current_hands=np.array([r4_hands[w]]),
        leads=np.array([r4_leads[w]]),
        game_round=4,
        game_score=np.array([r4_leads[w]]).reshape(-1, 1),
    )
    
    r6_leads, r6_hands, r5_score = next_round(
        current_hands=r5_hands, leads=r5_leads, game_round=5, game_score=r4_score
    )
    results = r5_score.reshape(r5_score.shape[0], 5)

    meta_results = np.zeros(results.shape[0], dtype=np.int64)

    for i in range(len(results)):
        meta_results[i] = np.sum(results[i]%2 + (r1_winner%2)+(r2_winner%2))>=3

if responder % 2 == 0:
    best_response = np.argmin(r3_response_res)
else:
    best_response = np.argmax(r3_response_res)
    
print(best_response)

0


In [ ]:
trick_played(r3_optimal, r4_hands[best_response])

array([[ -9,   0],
       [  0, -12],
       [  0, -10],
       [  0, -13]])

In [ ]:
r3_winner = r4_leads[best_response]

In [ ]:
r3_winner

np.int64(3)

In [ ]:
(r1_winner % 2) + (r2_winner % 2) + (r3_winner % 2)

np.int64(3)

In [ ]:
# some general notes:

# Should probably move the functions in here to tree search and figure out the best way to refactor without having to repeat code so much